
# Contents
### 1. The Iris CubeList and Cube objects
### 2. Plotting with iris
#### 2.1 Regional example
#### 2.2 Global example
---

# 1. The Iris CubeList and Cube objects

- NetCDF file provided contains air temperature and vertical velocity diagnostics on the hour for a 10-hour LFRic CRM simulation
- The dataset has coordinates (time, z, y, x) with shape (10, 141, 128, 128)

In [ ]:
import warnings
import iris
warnings.filterwarnings("ignore",module="iris.coords")
iris.FUTURE.date_microseconds = True

In [ ]:
# Download dataset from astro webserver
import os
if not os.path.exists("crm_cbl.nc"):
    os.system("curl -O https://www.star.bris.ac.uk/acorbett/devgroup/lfric_iris_intro/crm_cbl.nc")

In [ ]:
# Load the dataset
cubes = iris.load('crm_cbl.nc')

# If multiple diagnostics are present in the file, iris.load() will return a CubeList object containing multiple Cube objects.
# Each cube represents a different diagnostic and has its own metadata and data array.

In [ ]:
# Display the CubeList
cubes

In [ ]:
# Check if the cubes have lazy data (i.e., data that has not been loaded into memory yet)
for cube in cubes:
    print(f"Cube '{cube.name()}' has lazy data: {cube.has_lazy_data()}")

In [ ]:
# "Touch" the air_temperature data to load it into memory
data = cubes[0].data

cubes[0].has_lazy_data() # should now be false

In [ ]:
# Print the coordinates
cubes[0].coords()

# 2. Plotting with iris 

## 2.1 Regional example

In [ ]:
import iris.quickplot as qplt
# The iris quickplot module allows for quick visualisation of data in an iris cube

In [ ]:
# air_temperature example
air_temp_cube = cubes.extract('air_temperature')[0] # Extract the first (and only, in this case) cube with standard_name 'air_temperature'
upward_air_velocity_cube = cubes.extract('upward_air_velocity')[0] # Extract upward_air_velocity cube

qplt.pcolormesh(air_temp_cube[0, :, 64, :])  # Plot the first time step

In [ ]:
# Plot the initial and final states for the central column of the domain

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))

# Initial 
plt.subplot(1, 2, 1)
qplt.plot(air_temp_cube[0, :, 64, 64],air_temp_cube[0, :, 64, 64].coord("height")) # Plot the first time step
ax = plt.gca()

# Final
ax = plt.subplot(1, 2, 2)
qplt.plot(air_temp_cube[-1, :, 64, 64],air_temp_cube[-1, :, 64, 64].coord("height")) # Plot the last time step

plt.show()

In [ ]:
# Animation example

import iris.plot as iplt
import cartopy.crs as ccrs

# Ensure inline animation rendering in Jupyter
plt.rcParams['animation.html'] = 'html5'
from IPython.display import HTML

cube_iter = upward_air_velocity_cube[:,0:70,64,:].slices_over('time')

ani = iplt.animate(cube_iter, qplt.pcolormesh)
# Display inline in notebook
HTML(ani.to_jshtml(fps=5))

## 3.2 Global example

In [ ]:
fname = iris.sample_data_path('air_temp.pp') # Load sample dataset
cubes = iris.load(fname) 
cube = cubes[0] # Extract the first cube (air temperature)
cube

In [ ]:
plt.figure(figsize=(12, 8))

plt.subplot(1, 2, 1)
qplt.pcolormesh(cube)
ax = plt.gca()
ax.coastlines() # Add coastlines to the plot

ax = plt.subplot(1, 2, 2, projection=ccrs.Orthographic(central_longitude=0, central_latitude=51)) # Create a subplot with an orthographic projection centered on UK
qplt.pcolormesh(cube)
ax.coastlines()

plt.show()